# Parte 3 — Curadoria e Validação de IA Generativa
### Teste Técnico · Analista de Dados Pleno · FinLend (Lending Club)

A FinLend testa um agente de IA (LLM) que responde perguntas dos gestores sobre a
carteira de crédito. Antes de liberar a ferramenta, é preciso **validar se as
respostas fazem sentido**.

**Pergunta do gestor comercial:**
> *"Qual é o perfil de risco dos nossos clientes que tomam empréstimos para consolidação de dívidas?"*

**Resposta do agente (a ser validada):**
> *"Os clientes que tomam empréstimos para consolidação de dívidas representam **48%** da
> carteira total. A taxa de inadimplência desse segmento é de **12,3%**, ligeiramente
> abaixo da média geral de **14,1%**. O ticket médio é de **\$15.200** e a renda média
> anual é de **\$72.000**. A maioria dos clientes está classificada nas grades B e C
> (**62%**), indicando um perfil de risco moderado. A taxa de juros média é de **13,8%**."*

Este documento entrega: **(1)** validação quantitativa, **(2)** avaliação qualitativa,
**(3)** proposta de melhoria e **(4)** proposta de processo recorrente de validação.

*(Uso de IA assistiva declarado conforme o enunciado, como na Parte 2.)*

## 1. Validação quantitativa

Recalculamos cada número afirmado pelo agente diretamente sobre a base real
(a mesma amostra da Parte 2), para o segmento `debt_consolidation`.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("base_looker_finlend.csv", sep=';')
seg = df[df['finalidade'] == 'debt_consolidation']

# Recalcula cada afirmação
real = {
    '% da carteira (por qtd)'     : len(seg)/len(df)*100,
    'Inadimplência do segmento'   : seg['inadimplente'].mean()*100,
    'Inadimplência média geral'   : df['inadimplente'].mean()*100,
    'Ticket médio (US$)'          : seg['valor_emprestimo'].mean(),
    'Renda média anual (US$)'     : seg['renda_anual'].mean(),
    '% em grades B e C'           : seg['grade'].isin(['B','C']).mean()*100,
    'Taxa de juros média'         : seg['taxa_juros'].mean(),
}
agente = {
    '% da carteira (por qtd)'     : 48.0,
    'Inadimplência do segmento'   : 12.3,
    'Inadimplência média geral'   : 14.1,
    'Ticket médio (US$)'          : 15200,
    'Renda média anual (US$)'     : 72000,
    '% em grades B e C'           : 62.0,
    'Taxa de juros média'         : 13.8,
}

tab = pd.DataFrame({'Agente': agente, 'Real': real})
tab['Erro relativo %'] = ((tab['Agente'] - tab['Real']) / tab['Real'] * 100).round(1)
tab['Real'] = tab['Real'].round(2)
def veredito(e):
    e = abs(e)
    if e <= 5:  return 'OK'
    if e <= 15: return 'Impreciso'
    return 'INCORRETO'
tab['Veredito'] = tab['Erro relativo %'].apply(veredito)
tab

### Síntese da validação quantitativa

| Afirmação | Agente | Real | Veredito |
|---|---|---|---|
| % da carteira | 48% | **53,9%** | Impreciso (−11%) |
| Inadimplência do segmento | 12,3% | **7,5%** | ❌ **Incorreto (+64%)** |
| Inadimplência média geral | 14,1% | **7,0%** | ❌ **Incorreto (+101%)** |
| Ticket médio | \$15.200 | **\$16.595** | Impreciso (−8%) |
| Renda média anual | \$72.000 | **\$78.954** | Impreciso (−9%) |
| % em grades B e C | 62% | **59,4%** | OK (+4%) |
| Taxa de juros média | 13,8% | **13,5%** | ✅ OK (+2%) |

**Dos 7 números, apenas 2 estão corretos** (juros e % B/C). Os dois **mais críticos** —
as taxas de inadimplência — estão **inflados em ~2x**, justamente as métricas que um
gestor de risco usaria para decidir.

### O erro mais grave não é um número — é uma conclusão invertida

O agente afirma que o segmento (12,3%) está *"ligeiramente abaixo"* da média geral (14,1%).
Nos dados reais, a relação é **o oposto**:

In [ ]:
s = seg['inadimplente'].mean()*100
m = df['inadimplente'].mean()*100
print(f"Inadimplência do segmento : {s:.2f}%")
print(f"Inadimplência média geral : {m:.2f}%")
print(f"→ O segmento está {'ACIMA' if s>m else 'abaixo'} da média geral "
      f"(o agente afirmou o contrário).")

Consolidação de dívidas é, na verdade, **ligeiramente mais arriscada** que a média —
não menos. A conclusão narrativa do agente ("perfil de risco moderado, abaixo da média")
**inverte a realidade** e induziria o gestor a tratar o segmento como mais seguro do que é.

## 2. Avaliação qualitativa

Além dos números, como a resposta se comporta como ferramenta de decisão?

**a) Confiança sem fonte nem incerteza.** O agente cravou 7 números com tom assertivo,
sem citar de onde vieram, sem período de referência e sem qualquer margem. Em decisão de
crédito, um número sem procedência é um risco, não uma informação.

**b) Erros concentrados na métrica que mais importa.** Ironicamente, os números que o
agente mais erra (inadimplência) são os centrais para "perfil de risco" — o próprio tema
da pergunta. Acertar juros e errar risco pela metade é o pior padrão possível aqui.

**c) Mistura de acertos e erros no mesmo tom.** Alguns números estão certos (juros 13,8%)
e outros muito errados (inadimplência), todos apresentados com a mesma confiança. Isso é
perigoso: os acertos emprestam credibilidade aos erros, e o gestor não tem como distinguir.

**d) Precisão espúria.** Valores como "12,3%" e "48%" sugerem cálculo exato. Quando o
número real é 7,5% e 53,9%, a falsa precisão mascara que provavelmente são estimativas
(ou alucinações) e não consultas à base.

**e) Ausência de ressalva de contexto (conexão com a Parte 2).** Mesmo o "número real"
de inadimplência (7,5%) é uma **taxa crua**, que a Parte 2 mostrou ser subestimada por
**viés de maturação** — as safras recentes ainda não tiveram tempo de inadimplir. Um
agente confiável deveria sinalizar essa ressalva; o real de longo prazo será maior. Ou
seja: não basta o número bater com a base, ele precisa vir com o contexto que o torna
interpretável.

**f) Definições implícitas.** "Inadimplência" pode significar coisas diferentes (inclui
atraso de 30 dias? charge-off apenas?). O agente não explicita o critério, então nem dá
para saber se a divergência é de cálculo ou de definição.

## 3. Proposta de melhoria do agente

O problema de fundo é que o agente **"lembra" números em vez de consultá-los**. As
melhorias abaixo atacam a causa, não os sintomas:

1. **Grounding nos dados (text-to-SQL / RAG sobre a base).** O agente não deve gerar
   números de memória. Deve traduzir a pergunta em consulta (SQL ou camada semântica),
   executar sobre a base real e responder a partir do resultado. Isso, sozinho, elimina
   a maior parte das alucinações numéricas.

2. **Citar fonte e query.** Cada número deve vir acompanhado de origem: tabela, filtro
   aplicado, data de corte e, idealmente, a própria query. Torna a resposta auditável.

3. **Expressar incerteza e tamanho de amostra.** "7,5% de inadimplência (base: 53.924
   empréstimos, safras 2016–2018)". Números sem denominador não deveriam sair.

4. **Ressalvas automáticas de contexto.** Regras que injetam alertas conhecidos — ex.:
   ao falar de inadimplência por período, avisar sobre maturação de safra.

5. **Definições explícitas.** Fixar e exibir o critério de cada métrica (o que conta como
   inadimplente, como é calculado o ticket etc.).

6. **Guardrails de recusa.** Quando não houver dado suficiente, o agente deve dizer "não
   tenho base para responder com confiança" em vez de preencher a lacuna com estimativa.

## 4. Proposta de processo recorrente de validação

Validar uma vez não basta: o modelo muda, os dados mudam, as perguntas mudam. Proponho
um processo contínuo em cinco camadas:

**1. Conjunto de perguntas-referência (*golden set*).** Um catálogo de perguntas típicas
dos gestores com **respostas corretas calculadas por código** (a fonte de verdade), como
as deste notebook. É a régua contra a qual o agente é medido.

**2. Testes automatizados de regressão.** A cada nova versão do agente (ou do modelo
base), rodar o *golden set* automaticamente e comparar resposta vs verdade, com
**tolerância definida** (ex.: erro relativo ≤ 5% passa; > 15% falha e bloqueia o deploy).

**3. Métricas de qualidade monitoradas.** Acompanhar acurácia numérica, taxa de
alucinação (números sem lastro), cobertura (quantas perguntas ele responde com
grounding) e taxa de recusa apropriada.

**4. *Human-in-the-loop* para o que é novo ou sensível.** Perguntas fora do *golden set*,
ou que embasem decisões de alto impacto, passam por revisão de um analista antes de a
resposta ser considerada confiável — e viram novos itens do *golden set*.

**5. Monitoramento em produção + cadência.** Registrar perguntas e respostas reais,
amostrar periodicamente para auditoria, coletar *feedback* dos gestores (👍/👎) e
**revalidar o conjunto inteiro a cada atualização de dados ou troca de modelo**.

---

### Conclusão

A resposta do agente **não deve ser liberada como está**: erra 5 dos 7 números, infla a
métrica de risco em ~2x e inverte a conclusão qualitativa. Mais importante que corrigir
esta resposta é corrigir o **mecanismo** — ancorar o agente nos dados reais e cercá-lo de
um processo de validação contínuo. O caso reforça a lição da Parte 2: **um número
plausível pode estar errado**, e confiança não é o mesmo que correção.